# Fashion-MNIST transaction construction (TC1–TC7, CPU)

Notebook triển khai spec TC v2. Mặc định chọn cấu hình smoke; đặt `FASHION_TC_V2_CONFIG=configs/tc_v2_full.toml` để chạy toàn bộ 70.000 ảnh và 15 codebooks. Mỗi giai đoạn TC nằm trong cell riêng để có thể kiểm tra, chạy lại theo thứ tự và dừng ngay tại lỗi đầu tiên.

Luồng chính dùng raw full-batch K-means (B0), threshold cố định 29, TC6-S0. Q20 chỉ chạy khi chọn tường minh và sẽ báo lỗi nếu backend tương thích chưa được kết nối.


In [ ]:
using Pkg
using TOML

function find_repo_root(start=pwd())
    current = abspath(start)
    while true
        isfile(joinpath(current, "Project.toml")) && return current
        parent = dirname(current)
        parent == current && error("Could not find Project.toml from $start")
        current = parent
    end
end

const REPO_ROOT = find_repo_root()
Pkg.activate(REPO_ROOT)
using EclatFIM
const TC2 = EclatFIM.FashionTCv2
println("Julia ", VERSION, " | repo: ", REPO_ROOT)


## Cấu hình chạy

Notebook CPU chỉ nhận một trong hai config v2. Smoke dùng 20 ảnh được chọn cân bằng, không tạo train/validation split; full dùng toàn bộ train + test gốc trong một lượt TC.


In [ ]:
const CONFIG_OPTIONS = ("configs/tc_v2_smoke.toml", "configs/tc_v2_full.toml")
const CONFIG_RELATIVE_PATH = get(ENV, "FASHION_TC_V2_CONFIG", first(CONFIG_OPTIONS))
CONFIG_RELATIVE_PATH in CONFIG_OPTIONS || error("Unsupported TC v2 config: $CONFIG_RELATIVE_PATH")
const CONFIG_PATH = joinpath(REPO_ROOT, CONFIG_RELATIVE_PATH)
const CONFIG = TOML.parsefile(CONFIG_PATH)
CONFIG["execution"]["backend"] == "cpu" || error("This notebook is CPU-only")
println("Config: ", CONFIG_RELATIVE_PATH, " | mode: ", CONFIG["run"]["mode"],
    " | method: ", CONFIG["tc4_codebook"]["training_method"])


## Chuẩn bị dữ liệu và smoke assertions

Đọc cả hai split gốc và cấp `image_id` ổn định `1:70000`; labels chỉ được giữ làm metadata. Smoke chọn ảnh phân tầng từ tập gộp, không dùng nhãn trong descriptor, fit hay transaction.


In [ ]:
const RAW_DATA_DIR = joinpath(REPO_ROOT, CONFIG["dataset"]["data_dir"])
EclatFIM.download_fashion_mnist(RAW_DATA_DIR) # verifies official archive checksums
const RUN_STATE = TC2.prepare_run(CONFIG_PATH; repo_root=REPO_ROOT, data_dir=RAW_DATA_DIR)
const SMOKE_ASSERTIONS = TC2.run_unit_smoke_checks!()
@assert all(x -> x == "PASS", values(SMOKE_ASSERTIONS))
println("Loaded ", length(RUN_STATE.image_ids), " images; archive checksums verified")
SMOKE_ASSERTIONS


## TC1 — Trích patch 4×4

Xác nhận 49 patch/ảnh, không overlap/padding, thứ tự row-major và mapping tọa độ. Smoke kiểm tra tái tạo chính xác một ảnh từ các patch.


In [ ]:
TC1_RESULT = TC2.run_tc1!(RUN_STATE)
TC1_RESULT


## TC2 — Lọc foreground T2

Áp dụng đúng `any(pixel > 29)`. Không fit threshold và không chỉnh sửa pixel. Ảnh không còn patch được giữ lại để tạo transaction rỗng ở TC6.


In [ ]:
TC2_RESULT = TC2.run_tc2!(RUN_STATE)
TC2_RESULT


## TC3 — Raw pixel descriptors

Tạo vector Float32 16D từ `vec(patch) / 255f0`, giữ thứ tự column-major và mapping ảnh/patch/tọa độ.


In [ ]:
TC3_RESULT = TC2.run_tc3!(RUN_STATE)
TC3_RESULT


## TC4 — Codebook K-means và metrics

Mặc định fit tuần tự theo K × seed trên toàn bộ descriptors TC3, với K-means++, `maxiter=300`, `tol=1e-4`. Mỗi fit có checkpoint riêng. Ghi distortion, occupancy, cluster sizes, entropy, convergence và seed stability trên một mẫu cố định chỉ dùng để đánh giá.

Smoke config dùng một K/seed nhỏ để kiểm tra dây chuyền; chỉ full config tạo đủ 15 codebooks.


In [ ]:
TC4_RESULT = TC2.run_tc4!(RUN_STATE)
TC4_RESULT


## TC5 — Hard word assignment

Gán từng raw descriptor vào centroid gần nhất theo squared Euclidean, batch 16.384, tie-break chọn word ID nhỏ nhất. Lưu mapping patch có tọa độ cho từng codebook.


In [ ]:
TC5_RESULT = TC2.run_tc5!(RUN_STATE)
TC5_RESULT


## TC6 — Binary BoVW S0

Tạo một transaction mỗi ảnh bằng `sort(unique(word_ids))`. Xuất JSONL theo từng codebook; transaction rỗng vẫn có `tid` và `image_id`. Labels và tọa độ không xuất hiện trong `items`.


In [ ]:
TC6_RESULT = TC2.run_tc6!(RUN_STATE)
TC6_RESULT


## TC7 — Reload và kiểm tra integrity

Đọc lại codebook, assignments, mapping và transactions sau export. Kiểm tra đủ ảnh/TID/transactions, transaction rỗng, mã từ vựng và denominator support 70.000 (hoặc số ảnh ở smoke). Chỉ ghi trạng thái `COMPLETED` khi mọi kiểm tra read-back đạt.


In [ ]:
TC7_RESULT = TC2.run_tc7!(RUN_STATE)
TC7_RESULT


## Kết quả

Artifact mỗi codebook nằm trong `output_root` theo `method/K/seed`, kèm config snapshot, metrics, checkpoint và integrity report. Notebook không chạy EDA2/Pilot Eclat, không chọn K cuối cùng và không triển khai S1/S2.


In [ ]:
@assert RUN_STATE.stage == 7
@assert all(r -> r.status == "COMPLETED", TC7_RESULT.runs)
println("TC v2 notebook cells completed: ", RUN_STATE.stage, "/7")
println("Output: ", RUN_STATE.output_root)
